# **MÓDULO 3: SISTEMA DE RECOMENDACIÓN**
## IONClinics & Universidad Complutense de Madrid

## **Pre-Work Checklist: Ejecutar antes de comenzar**

Esta etapa inicial prepara el entorno de trabajo para la ejecución del sistema. Aquí se cargan las librerías necesarias, se configuran las rutas a las bases de datos y archivos requeridos, y se establecen las sesiones de Spark. Este paso es fundamental para garantizar que todos los procesos posteriores se ejecuten de manera fluida y sin errores relacionados con la configuración o dependencias del entorno.



In [ ]:
from google.colab import drive
import os, sys
drive.mount('/content/drive')
print(os.getcwd())


Mounted at /content/drive
/content


In [ ]:
os.chdir('/content/drive/MyDrive/Data Lake IONClinics')

In [ ]:
from configparser import ConfigParser
from pathlib import Path
import requests
import urllib
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import ast
import uuid
from sklearn.preprocessing import LabelEncoder
import random
from sklearn.model_selection import train_test_split
from collections import deque
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install --upgrade pyspark
!pip install py4j
!pip install --upgrade openai
import os
import sys
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, explode, from_json, substring, split, udf
from pyspark.sql.types import StringType, StructType, StructField, IntegerType, ArrayType
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.utils import AnalysisException
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn import preprocessing, model_selection
import torch.nn as nn
from sklearn.metrics import mean_squared_error
from collections import defaultdict


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,927 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,901 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,722

### **Parameter setting**

PATHS

In [ ]:
# Path for storage
database_dir = Path('/content/drive/MyDrive/Data Lake IONClinics/DataBase')
dtset_dir = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo')
dtset_dir_parquet = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark')

In [ ]:
# Base de datos de protocolos con parámetros mapeados
df_map = pd.read_excel(database_dir / 'Protocolos/BBDD_mapeado_mayo_2025.xlsx')

# Base de datos de protocolos con parámetros orginales
#df_no_map = pd.read_excel(database_dir / 'Protocolos/BBDD_label_encoder_diciembre_2024.xlsx')
df_no_map = pd.read_excel(database_dir / 'Protocolos/BBDD_label_encoder_mayo_2025.xlsx')

## Pre-Train dataset load

# 1- Población Respuesta para comportamiento aleatorio
#poblacion_respuesta_completa = pd.read_excel(database_dir / 'Poblaciones/poblacion_respuesta_completa.xlsx')

# 2- Población Sesgada para comportamiento que se espera
poblacion_sesgada_completa = pd.read_excel(database_dir / 'Poblaciones/poblacion_sesgada_completa.xlsx')

### **Spark Session**

In [ ]:
from pyspark.sql import SparkSession
import findspark

# Initialize Spark
findspark.init()

# Create or get the SparkSession with custom configurations
spark = SparkSession.builder \
    .appName("tDCS_recomendador") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Create an RDD with increased partitions
rdd = spark.sparkContext.parallelize(range(100), numSlices=8)  # Increase the number of partitions

# Get the number of partitions
num_partitions = rdd.getNumPartitions()
print("Number of partitions (indirect indicator of workers):", num_partitions)


Number of partitions (indirect indicator of workers): 8


In [ ]:
spark

In [ ]:
dtset_dir_parquet = Path(dtset_dir_parquet)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
hdfs_dir_parquet = spark._jvm.org.apache.hadoop.fs.Path(dtset_dir_parquet.as_posix())
if not fs.exists(hdfs_dir_parquet):
    fs.mkdirs(hdfs_dir_parquet)

### Mapping process

In [ ]:
unique_values_current = df_no_map['Current (mA)'].unique()
unique_values_current.sort()
current_mapping = {current: idx + 1 for idx, current in enumerate(unique_values_current)}
print(current_mapping)

{'0,5 - 2,2': 1, '0.56': 2, '0.7': 3, '1.0': 4, '1.2': 5, '1.4': 6, '1.5': 7, '1.6': 8, '2.0': 9, '2.5': 10}


In [ ]:
# Mapping dictionaries
electrode_mapping = {
    'Not specified': 0,
    'others': 1,
    'C3/C4': 2,
    'F3/F4': 3,
    'Fp1/Fp2': 4,
    'F7/F8': 5,
    'P3/P4': 6,
    'Cz': 7,
    'Supraorbital region': 8,
    'T3/T4': 9,
    'O1/O2': 10,
    'Fz': 11,
    'FpZ': 12,
    'Oz': 13,
    'Anatomical area': 14
}

symptom_keyword_mapping = {
    'Not specified': 0,
    'others': 1,
    'motor': 2,
    'depression': 3,
    'chronic pain': 4,
    'aphasia': 5,
    'complex symptoms associated with fibromyalgia': 6,
    'cognitive function': 7,
    'neuropathic pain': 8,
    'pain associated with fibromyalgia': 9,
    'complex symptoms associated with schizophrenia': 10,
    'memory': 11
}

pathology_keyword_mapping = {
    'Not specified': 0,
    'others': 1,
    'stroke': 2,
    'pain': 3,
    'depression': 4,
    'schizophrenia': 5,
    'spinal cord injury': 6,
    'fibromyalgia': 7,
    'cognitive decline': 8,
    'knee osteoarthritis': 9,
    'mental-health disorder': 10,
    'cancer': 11
}

modality_mapping = {
    'Not specified': 0,
    'others': 1,
    'Anodal': 2,
    'Bilateral': 3,
    'High-definition': 4,
    'Cathodal': 5,
    'Multifocal': 6,
    'TMS': 7,
    'tDCS combined': 8
}

evidence_mapping = {
    'A': 4,
    'B': 3,
    'C': 2,
    'D': 1
}


reverse_current_mapping = {v: k for k, v in current_mapping.items()}
reverse_symptom_keyword_mapping = {v: k for k, v in symptom_keyword_mapping.items()}
reverse_pathology_keyword_mapping = {v: k for k, v in pathology_keyword_mapping.items()}
reverse_electrode_mapping = {v: k for k, v in electrode_mapping.items()}
reverse_modality_mapping = {v: k for k, v in modality_mapping.items()}


## **Sección de Entrenamiento**

Esta sección del código se encarga de entrenar los modelos correspondientes a cada una de las lógicas de recomendación implementadas, y de guardar los pesos resultantes para su uso posterior. Es importante ejecutar esta parte únicamente cuando se desee actualizar o ajustar los modelos con nuevos datos. En caso de que ya se cuenten con modelos previamente entrenados y únicamente se quiera utilizar el sistema de recomendación sin realizar un nuevo entrenamiento, esta sección puede omitirse y se puede continuar directamente con la sección de despliegue.



### Modelo Recomendador #1: **Mecanismo de atención por Sintomatología**

Este modelo utiliza un mecanismo de atención para priorizar los protocolos en función de la sintomatología del paciente. La arquitectura está diseñada para identificar patrones relevantes entre los síntomas reportados y los parámetros de protocolos previamente utilizados, enfocándose en aquellos que han mostrado mayor efectividad para casos similares. Esta lógica permite una recomendación más personalizada y centrada en la condición clínica específica del paciente.

In [ ]:
### **Neural Network with Attention Mechanism**
class AdaptiveTDCSNNWithAttention(nn.Module):
    def __init__(self, num_protocols, input_dim, priority_weights=None):
        super(AdaptiveTDCSNNWithAttention, self).__init__()
        self.input_dim = input_dim

        # Initialize attention weights based on priority
        if priority_weights is None:
            # Default priority: symptom > pathology > age > gender
            priority_weights = [0.6, 0.2, 0.1, 0.1]
            expanded_weights = (
                [priority_weights[0]] +  # Symptom
                [priority_weights[1]] +  # Pathology
                [priority_weights[2]] * 4 +  # Age groups
                [priority_weights[3]] * 3  # Gender groups
            )
        else:
            expanded_weights = priority_weights

        self.attention_weights = nn.Parameter(torch.tensor(expanded_weights, dtype=torch.float32), requires_grad=True)

        # Fully connected layers
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_protocols)

    def forward(self, x):
        # Apply attention weights to the input
        attention_scores = torch.softmax(self.attention_weights, dim=0)
        x = x * attention_scores

        # Forward pass through the network
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

class TDCSRecommenderWithAttention:
    def __init__(self, model, label_encoder, df, pathology_keyword_mapping, symptom_keyword_mapping):
        self.model = model
        self.label_encoder = label_encoder
        self.df = df
        self.pathology_keyword_mapping = pathology_keyword_mapping
        self.symptom_keyword_mapping = symptom_keyword_mapping
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)

    def map_age_to_columns(self, age):
        age_groups = {'Kid_0_5': 0, 'Youth_6_17': 0, 'Adult_18_59': 0, 'Elderly_60_100': 0}
        if 0 <= age <= 5:
            age_groups['Kid_0_5'] = 1
        elif 6 <= age <= 17:
            age_groups['Youth_6_17'] = 1
        elif 18 <= age <= 59:
            age_groups['Adult_18_59'] = 1
        elif 60 <= age <= 100:
            age_groups['Elderly_60_100'] = 1
        return age_groups

    def map_gender_to_columns(self, gender):
        gender_columns = {'Males': 0.0, 'Females': 0.0, 'No_gender': 0.0}
        if gender.lower() == 'male':
            gender_columns['Males'] = 1.0
        elif gender.lower() == 'female':
            gender_columns['Females'] = 1.0
        else:
            gender_columns['No_gender'] = 1.0
        return gender_columns

    def calculate_weighted_feedback(self, feedback_history, decay_factor=0.8):
        min_feedback, max_feedback = 1, 5
        normalized_feedback = [(f - min_feedback) / (max_feedback - min_feedback) for f in feedback_history]
        feedback_weights = [decay_factor ** i for i in range(len(normalized_feedback))]
        weighted_feedback = sum(f * w for f, w in zip(normalized_feedback[::-1], feedback_weights))
        return weighted_feedback / sum(feedback_weights)

    def recommend_protocol(self, inputs, feedback_history_patient, feedback_history_clinician, rejected_protocols, previous_protocol):

        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if previous_protocol is not None:
            weighted_patient_feedback = self.calculate_weighted_feedback(feedback_history_patient)
            weighted_clinician_feedback = self.calculate_weighted_feedback(feedback_history_clinician)

            if weighted_patient_feedback >= 0.5:
                return previous_protocol

            rejected_protocols.add(previous_protocol)

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(inputs)
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        while protocol_label in rejected_protocols:
            outputs[0, predicted_label_idx] = -float('inf')
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        return protocol_label

    def generate_training_data_from_synthetic(self, synthetic_dataframes, num_sessions=10,training_poblacion_sesgada=False, training_poblacion_respuesta=False):
        training_data = []
        synthetic_dataframes = synthetic_dataframes.reset_index(drop=True)
        synthetic_dataframes['Response_profile'] = synthetic_dataframes['Response_profile'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        for idx , user in synthetic_dataframes.iterrows():
            pathology_map = self.pathology_keyword_mapping[user['Pathology']]
            symptom_map = self.symptom_keyword_mapping[user['Symptom'].lower()]
            age_group_map = self.map_age_to_columns(user['Age'])
            gender_map = self.map_gender_to_columns(user['Gender'])

            inputs = [symptom_map, pathology_map] + list(age_group_map.values()) + list(gender_map.values())
            patient_inputs_tensor = torch.tensor(inputs, dtype=torch.float32)

            feedback_history_patient = []
            feedback_history_clinician = []
            rejected_protocols = set()

            for session in range(num_sessions):

                protocol_label = self.recommend_protocol(
                    patient_inputs_tensor.unsqueeze(0),
                    feedback_history_patient,
                    feedback_history_clinician,
                    rejected_protocols,
                    None if session == 0 else protocol_label
                )

                if training_poblacion_sesgada:
                    feedback_score = self.check_protocol_feedback(protocol_label, synthetic_dataframes, idx)

                else:
                    feedback_score = int(user['Response_profile'][session])

                feedback_history_patient.append(feedback_score)
                feedback_history_clinician.append(feedback_score)


                training_data.append((idx,patient_inputs_tensor, protocol_label, feedback_score))


                if session % 10 == 0 and session !=0 :
                    self.train_model_with_synthetic_data(training_data,synthetic_dataframes)

        return

    def train_model_with_synthetic_data(self, training_data, synthetic_dataframes, learning_rate=0.0001):

        self.model.train()
        batch_size = min(64, len(training_data))
        batch = random.sample(training_data, batch_size)
        epoch_loss = 0

        for idx, inputs, target,_ in batch:
            self.optimizer.zero_grad()
            inputs = inputs.unsqueeze(0) if inputs.dim() == 1 else inputs
            outputs = self.model(inputs)
            predicted_label = torch.argmax(outputs, dim=1).item()
            feedback_score = self.check_protocol_feedback(predicted_label, synthetic_dataframes, idx)
            loss = self.criterion(outputs, torch.tensor([target]))

            if feedback_score >= 4:
                penalty_factor = 0.2  # No scaling for good feedback
            else:
                penalty_factor = 1.1 + ((4 - feedback_score) / 3) ** 2  # e.g., 1.0 to ~2.78


            loss *= penalty_factor
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()

    def incremental_training(self, synthetic_dataframes, num_epochs_per_stage=10, learning_rate=0.001,training_poblacion_respuesta=False, training_poblacion_sesgada=False):
          if training_poblacion_respuesta:
              print('Entrenando a población respuesta')
              self.generate_training_data_from_synthetic(synthetic_dataframes, training_poblacion_respuesta=True)
              i = 'poblacion respuesta'


          elif training_poblacion_sesgada:
              print('Entrenando a población sesgada')
              self.generate_training_data_from_synthetic(synthetic_dataframes, training_poblacion_sesgada=True)
              i = 'población sesgada'

          print(f"Entrenamiento finalizado para {i}")
          print("----------------------------------------------------------")

    def check_protocol_feedback(self, recommended_protocol, synthetic_dataframes, patient_index):

        """
        Función de comprobación de protocolos.

        Arguments:
            - recommended_protocol (int): El corpusid del protocolo recomendado.
            - df_SintUsers (df): DataFrame con los datos de los pacientes de entrenamiento.
            - patient_index (int): El índice del paciente en el DataFrame.

        Returns:
            int: Feedback del paciente al protocolo (1-5).
        """
        synthetic_dataframes['Protocols'] = synthetic_dataframes['Protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        synthetic_dataframes['Exact_protocols'] = synthetic_dataframes['Exact_protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        # Extraemos los protocolos válidos y exactos del paciente
        valid_protocols = synthetic_dataframes.iloc[patient_index]['Protocols']
        extra_valid_protocols = synthetic_dataframes.iloc[patient_index]['Exact_protocols']

        # Asegurar que los valores son listas
        if not isinstance(valid_protocols, list):
            raise ValueError(f"Expected a list for 'Protocols', but got {type(valid_protocols)}")
        if not isinstance(extra_valid_protocols, list):
            raise ValueError(f"Expected a list for 'Exact_protocols', but got {type(extra_valid_protocols)}")

        # Si el protocolo recomendado está entre los válidos -- trata el síntoma
        if recommended_protocol in valid_protocols:
            # Si el protocolo trata también la patología del paciente
            if recommended_protocol in extra_valid_protocols:

                return 5  # Feedback positivo con patología y síntoma en común
            else:
                return 4  # Feedback positivo, pero sólo ajusta al síntoma

        # Si no está en la lista de protocolos válidos -- feedback negativo
        else:
            return random.choice([1, 2])

def pretrain_model(model, X, y, num_epochs=100, learning_rate=0.001):
    """
    Pre-trains the model on the original dataframe to establish a baseline before
    incorporating synthetic training data and feedback-weighted training.
    """

    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y.numpy()), y=y.numpy())
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X)  # Forward pass
        loss = criterion(outputs, y)  # Calculate loss
        loss.backward()  # Backward pass (gradient calculation)
        optimizer.step()  # Update weights

        if (epoch + 1) % 10 == 0:
            print(f"Pre-training Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

**Pre-entrenamiento + Pacientes digitales**

In [ ]:
input_columns = ['Symptom_map','Pathology_map', 'Kid_0_5', 'Youth_6_17', 'Adult_18_59',
                 'Elderly_60_100', 'Males', 'Females', 'No_gender']

protocol_columns = ['Current_map', 'Duration (min)', 'Times_per_day', 'Days_per_week',
                    'Cathode_standarized_map', 'Anode_standarized_map']

df_map[input_columns] = df_map[input_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
df_map[protocol_columns] = df_map[protocol_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode Protocol_Label to numeric labels
label_encoder = LabelEncoder()
df_map['Protocol_Label_encoded'] = label_encoder.fit_transform(df_map['Protocol_Label'])

X = torch.tensor(df_map[input_columns].values, dtype=torch.float32)
y = torch.tensor(df_map['Protocol_Label_encoded'].values, dtype=torch.long)

num_protocols = len(label_encoder.classes_)
input_dim = len(input_columns)
## PRE-TRAIN + POBLACION_RESPUESTA + POBLACIÓN_SESGADA
model_with_attention = AdaptiveTDCSNNWithAttention(num_protocols, input_dim)
recommender_with_attention = TDCSRecommenderWithAttention(
    model_with_attention, label_encoder, df_map, pathology_keyword_mapping, symptom_keyword_mapping
)

save_path = "/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/"
os.makedirs(save_path, exist_ok=True)  # Ensure the directory exists

def save_final_model_weights(model, filename="tdcs_model_weights_attention.pth"):
    """ Saves the final model weights after all training stages. """
    file_path = os.path.join(save_path, filename)
    torch.save(model.state_dict(), file_path)
    print(f"Final model weights saved at: {file_path}")

pretrain_model(model_with_attention, X, y, num_epochs=200, learning_rate=0.001)

train_val_splits_poblacion_sesgada = {}
train_val_splits_poblacion_respuesta = {}

train_df_sesgada, val_df_sesgada = train_test_split(poblacion_sesgada_completa, test_size=0.2, random_state=42)
train_val_splits_poblacion_sesgada = {"train": train_df_sesgada, "val": val_df_sesgada}

#train_df_respuesta, val_df_respuesta = train_test_split(poblacion_respuesta_completa, test_size=0.2, random_state=42)
#train_val_splits_poblacion_respuesta = {"train": train_df_respuesta, "val": val_df_respuesta}

synthetic_training_dataframe_sesgada = train_val_splits_poblacion_sesgada["train"]
#synthetic_training_dataframe_respuesta = train_val_splits_poblacion_respuesta["train"]

#recommender_with_attention.incremental_training(synthetic_training_dataframe_respuesta, num_epochs_per_stage=3,training_poblacion_respuesta=True)
recommender_with_attention.incremental_training(synthetic_training_dataframe_sesgada, num_epochs_per_stage=7,training_poblacion_sesgada=True)

save_final_model_weights(model_with_attention) # Uncomment to save weights

Pre-training Epoch [10/200], Loss: 5.1164
Pre-training Epoch [20/200], Loss: 5.0878
Pre-training Epoch [30/200], Loss: 5.0090
Pre-training Epoch [40/200], Loss: 4.8500
Pre-training Epoch [50/200], Loss: 4.6445
Pre-training Epoch [60/200], Loss: 4.4276
Pre-training Epoch [70/200], Loss: 4.2048
Pre-training Epoch [80/200], Loss: 3.9811
Pre-training Epoch [90/200], Loss: 3.7670
Pre-training Epoch [100/200], Loss: 3.5749
Pre-training Epoch [110/200], Loss: 3.4111
Pre-training Epoch [120/200], Loss: 3.2643
Pre-training Epoch [130/200], Loss: 3.1276
Pre-training Epoch [140/200], Loss: 3.0033
Pre-training Epoch [150/200], Loss: 2.8895
Pre-training Epoch [160/200], Loss: 2.7800
Pre-training Epoch [170/200], Loss: 2.6721
Pre-training Epoch [180/200], Loss: 2.5643
Pre-training Epoch [190/200], Loss: 2.4589
Pre-training Epoch [200/200], Loss: 2.3595
Entrenando a población sesgada
Entrenamiento finalizado para población sesgada
----------------------------------------------------------
Final model

### Modelo Recomendador #2: **Aprendizaje por Refuerzo**

Este modelo implementa una lógica basada en aprendizaje por refuerzo, donde el sistema aprende a optimizar sus recomendaciones a lo largo del tiempo mediante la retroalimentación recibida de cada sesión terapéutica. A través de un proceso iterativo, el agente de recomendación ajusta sus decisiones para maximizar la efectividad del tratamiento, considerando tanto la evolución del paciente como la respuesta obtenida en sesiones anteriores. Esta lógica es ideal para escenarios donde se busca una adaptación dinámica y progresiva del protocolo.

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)  # Q-values for each protocol

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)  # Q-values for each action
        return x

class DQNAgent:
    def __init__(self, state_dim, action_dim, label_encoder, pathology_mapping, symptom_mapping,
                 learning_rate=0.001, gamma=0.7, memory_size=20000, batch_size=128):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.label_encoder = label_encoder
        self.pathology_mapping = pathology_mapping
        self.symptom_mapping = symptom_mapping

        # Initialize Q-network and target network
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.target_model.load_state_dict(self.model.state_dict())  # Sync target model
        self.last_feedback = 3.0

        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.MSELoss()

        # Experience replay buffer
        self.memory = deque(maxlen=memory_size)
        self.batch_size = batch_size

        # Hyperparameters
        self.gamma = gamma
        self.update_target_every = 3  # Update target network every X steps
        self.steps = 0

        self.loss_history = []  # Stores loss per batch

    def store_experience(self, state, action, reward, next_state):
        """Stores experiences in the replay buffer"""
        self.memory.append((state, action, reward, next_state))

    def map_age_to_columns(self, age):
        age_groups = {'Kid_0_5': 0, 'Youth_6_17': 0, 'Adult_18_59': 0, 'Elderly_60_100': 0}
        if 0 <= age <= 5:
            age_groups['Kid_0_5'] = 1
        elif 6 <= age <= 17:
            age_groups['Youth_6_17'] = 1
        elif 18 <= age <= 59:
            age_groups['Adult_18_59'] = 1
        elif 60 <= age <= 100:
            age_groups['Elderly_60_100'] = 1
        return age_groups

    def map_gender_to_columns(self, gender):
        gender_columns = {'Males': 0.0, 'Females': 0.0, 'No_gender': 0.0}
        if gender.lower() == 'male':
            gender_columns['Males'] = 1.0
        elif gender.lower() == 'female':
            gender_columns['Females'] = 1.0
        else:
            gender_columns['No_gender'] = 1.0
        return gender_columns

    def select_action(self, state, previous_action,rejected_protocols,feedback_history_patient,feedback_history_clinician,generate_training=False):

        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if previous_action is not None:
            if generate_training:
                if feedback_history_patient[-1] > 5:
                   action_idx = self.label_encoder.transform([previous_action])[0]
                   return int(action_idx), previous_action

                else:
                    rejected_protocols.add(previous_action)

            else: # Manual Checking
                if (feedback_history_patient[-1] + feedback_history_clinician[-1])/2 > 3:
                    action_idx = self.label_encoder.transform([previous_action])[0]
                    return int(action_idx), previous_action
                else:
                    rejected_protocols.add(previous_action)

        self.model.eval()
        with torch.no_grad():
              q_values = self.model(torch.FloatTensor(state).unsqueeze(0))
              action_idx = torch.argmax(q_values).item()
              protocol_label = self.label_encoder.inverse_transform([action_idx])[0]

        while protocol_label in rejected_protocols:
              q_values[0, action_idx] = -float('inf')
              action_idx = torch.argmax(q_values).item()
              protocol_label = self.label_encoder.inverse_transform([action_idx])[0]

        return action_idx, protocol_label

    def train_RL(self,manual_comparison=False):
        """Trains the DQN using replay memory with constrained Q-value updates and feedback-aware loss weighting."""

        if len(self.memory) < self.batch_size:
            batch_size = len(self.memory)  # Use all available memory if smaller
        else:
            batch_size = self.batch_size

        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states = zip(*batch)

        states = torch.stack(states)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.stack(next_states)

        #Compute Q-values for selected actions
        q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            best_actions = self.target_model(next_states).argmax(1, keepdim=True)
            next_q_values = self.target_model(next_states).gather(1, best_actions).squeeze(1)

        # Reward Scaling
        structured_rewards = -2 + 4 * (rewards - 1) / 4  # Now in range [-2, 2]

        # Compute target Q-values with constraints
        target_q_values = structured_rewards + self.gamma * next_q_values

        # Feedback-Weighted Loss (low feedback → higher impact on training)
        loss_weights = torch.exp(-structured_rewards)  # More penalty for bad feedback

        if hasattr(self, 'last_feedback') and isinstance(self.last_feedback, (int, float)):
            if self.last_feedback <= 2:
                #loss_weights *= 1.5  # More weight (higher penalty)
                loss_weights *= 3  # More weight (higher penalty)

        weighted_loss = loss_weights * (q_values - target_q_values) ** 2
        loss = weighted_loss.mean()
        if manual_comparison:
            print(f"Loss: {loss.item():.4f}")

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()

        self.loss_history.append(loss.item())

        # Update target network periodically
        if manual_comparison:
            self.target_model.load_state_dict(self.model.state_dict())
        else:
            if self.steps % self.update_target_every == 0:
                self.target_model.load_state_dict(self.model.state_dict())

        self.steps += 1


    def generate_training_data_from_synthetic(self, synthetic_dataframes, num_sessions=10,training_poblacion_sesgada=False, training_poblacion_respuesta=False):
          synthetic_dataframes = synthetic_dataframes.reset_index(drop=True)
          synthetic_dataframes['Response_profile'] = synthetic_dataframes['Response_profile'].apply(
              lambda x: ast.literal_eval(x) if isinstance(x, str) else x
          )

          for idx, user in synthetic_dataframes.iterrows():
              pathology_map = self.pathology_mapping[user['Pathology']]
              symptom_map = self.symptom_mapping[user['Symptom'].lower()]
              age_group_map = self.map_age_to_columns(user['Age'])
              gender_map = self.map_gender_to_columns(user['Gender'])

              state = [symptom_map, pathology_map] + list(age_group_map.values()) + list(gender_map.values())
              state_tensor = torch.tensor([state], dtype=torch.float32).squeeze(0)

              feedback_history_patient = []
              feedback_history_clinician = []
              rejected_protocols = set()
              feedback_score = 0

              for session in range(num_sessions):

                  if session == 0:
                    self.target_model.load_state_dict(self.model.state_dict())


                  action_idx, protocol_label = self.select_action(state=state_tensor,
                                                                  previous_action= None if session == 0 else protocol_label,
                                                                  rejected_protocols=rejected_protocols,
                                                                  feedback_history_patient=feedback_history_patient,
                                                                  feedback_history_clinician=feedback_history_clinician,
                                                                  generate_training= True)

                  if training_poblacion_sesgada:
                      feedback_score = self.check_protocol_feedback(protocol_label, synthetic_dataframes, idx)

                  else:
                      feedback_score = int(user['Response_profile'][session])

                  feedback_history_patient.append(feedback_score)
                  feedback_history_clinician.append(feedback_score)

                  self.last_feedback = feedback_score

                  next_state = state_tensor
                  self.store_experience(state_tensor, action_idx, feedback_score, next_state)


                  self.train_RL()

          return

    def incremental_training(self, synthetic_dataframes, num_epochs_per_stage=1, learning_rate=0.0001,
                         training_poblacion_respuesta=False, training_poblacion_sesgada=False):

        if training_poblacion_respuesta:
            print('Entrenando a población respuesta')
            for epoch in range(num_epochs_per_stage):
                self.generate_training_data_from_synthetic(synthetic_dataframes,training_poblacion_sesgada=False, training_poblacion_respuesta=True)
                i = 'poblacion respuesta'


        elif training_poblacion_sesgada:
            print('Entrenando a población sesgada')
            for epoch in range(num_epochs_per_stage):
                self.generate_training_data_from_synthetic(synthetic_dataframes,training_poblacion_sesgada=True, training_poblacion_respuesta=False)
                i = 'población sesgada'


        print(f"Entrenamiento finalizado para {i}")
        print("----------------------------------------------------------")

    def check_protocol_feedback(self, recommended_protocol, synthetic_dataframes, patient_index):

        """
        Función de comprobación de protocolos.

        Arguments:
            - recommended_protocol (int): Label protocolo recomendado.
            - df_SintUsers (df): DataFrame con los datos de los pacientes de entrenamiento.
            - patient_index (int): El índice del paciente en el DataFrame.

        Returns:
            int: Feedback del paciente al protocolo (1-5).
        """
        synthetic_dataframes['Protocols'] = synthetic_dataframes['Protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        synthetic_dataframes['Exact_protocols'] = synthetic_dataframes['Exact_protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        valid_protocols = synthetic_dataframes.iloc[patient_index]['Protocols']
        extra_valid_protocols = synthetic_dataframes.iloc[patient_index]['Exact_protocols']

        # Asegurar que los valores son listas
        if not isinstance(valid_protocols, list):
            raise ValueError(f"Expected a list for 'Protocols', but got {type(valid_protocols)}")
        if not isinstance(extra_valid_protocols, list):
            raise ValueError(f"Expected a list for 'Exact_protocols', but got {type(extra_valid_protocols)}")

        # Si el protocolo recomendado está entre los válidos -- trata el síntoma
        if recommended_protocol in valid_protocols:
            # Si el protocolo trata también la patología del paciente
            if recommended_protocol in extra_valid_protocols:
                counter_exact_protocols =+1
                return 5.0  # Feedback positivo con patología y síntoma en común
            else:
                counter_valid_protocols =+ 1
                return 4.0  # Feedback positivo, pero sólo ajusta al síntoma

        # Si no está en la lista de protocolos válidos -- feedback negativo
        else:
            counter_invalid_protocols =+ 1
            return random.choice([1.0, 2.0])


def pretrain_model(agent, X, y, num_epochs=100):
    """Pre-trains the model using the real patient-protocol dataset."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(agent.model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        agent.model.train()
        agent.optimizer.zero_grad()

        outputs = agent.model(X)
        loss = criterion(outputs, y)
        loss.backward()
        agent.optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Pre-training Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
    print("✅ Pretraining Completed")
    #torch.save(agent.model.state_dict(), "pretrained_model.pth")  # Save pre-trained model
    #print("💾 Pretrained Model Saved Successfully!")


**Pre-entrenamiento + Pacientes digitales**

In [ ]:
input_columns = ['Symptom_map','Pathology_map', 'Kid_0_5', 'Youth_6_17', 'Adult_18_59',
                 'Elderly_60_100', 'Males', 'Females', 'No_gender']

protocol_columns = ['Current_map', 'Duration (min)', 'Times_per_day', 'Days_per_week',
                    'Cathode_standarized_map', 'Anode_standarized_map']

df_map[input_columns] = df_map[input_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
df_map[protocol_columns] = df_map[protocol_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode Protocol_Label to numeric labels
label_encoder = LabelEncoder()
df_map['Protocol_Label_encoded'] = label_encoder.fit_transform(df_map['Protocol_Label'])

X = torch.tensor(df_map[input_columns].values, dtype=torch.float32)
y = torch.tensor(df_map['Protocol_Label_encoded'].values, dtype=torch.long)

state_dim = len(input_columns)
action_dim = len(label_encoder.classes_)
recommender_RL = DQNAgent(state_dim, action_dim, label_encoder, pathology_keyword_mapping, symptom_keyword_mapping)

save_path = "/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/"
os.makedirs(save_path, exist_ok=True)  # Ensure the directory exists

def save_final_model_weights(agent, filename="tdcs_model_weights_RL.pth"):
    """ Saves the final model weights after all training stages. """
    file_path = os.path.join(save_path, filename)
    torch.save(agent.model.state_dict(), file_path)
    print(f"Final model weights saved at: {file_path}")

print("\n--- Iniciando Pre-Entrenamiento ---")
pretrain_model(recommender_RL, X, y, num_epochs=200)

train_val_splits_poblacion_respuesta = {}
train_val_splits_poblacion_sesgada = {}

train_df, val_df = train_test_split(poblacion_sesgada_completa, test_size=0.2, random_state=42)
train_val_splits_poblacion_sesgada = {"train": train_df, "val": val_df}

synthetic_training_dataframe_sesgada = train_val_splits_poblacion_sesgada["train"]

recommender_RL.incremental_training(synthetic_training_dataframe_sesgada, num_epochs_per_stage=2,training_poblacion_sesgada=True)

save_final_model_weights(recommender_RL) #Uncomment to rewrite the model weights


--- Iniciando Pre-Entrenamiento ---
Pre-training Epoch [10/200], Loss: 4.9826
Pre-training Epoch [20/200], Loss: 4.7698
Pre-training Epoch [30/200], Loss: 4.5806
Pre-training Epoch [40/200], Loss: 4.4523
Pre-training Epoch [50/200], Loss: 4.3128
Pre-training Epoch [60/200], Loss: 4.1503
Pre-training Epoch [70/200], Loss: 3.9699
Pre-training Epoch [80/200], Loss: 3.7833
Pre-training Epoch [90/200], Loss: 3.6073
Pre-training Epoch [100/200], Loss: 3.4434
Pre-training Epoch [110/200], Loss: 3.2909
Pre-training Epoch [120/200], Loss: 3.1470
Pre-training Epoch [130/200], Loss: 3.0075
Pre-training Epoch [140/200], Loss: 2.8673
Pre-training Epoch [150/200], Loss: 2.7205
Pre-training Epoch [160/200], Loss: 2.5710
Pre-training Epoch [170/200], Loss: 2.4254
Pre-training Epoch [180/200], Loss: 2.2887
Pre-training Epoch [190/200], Loss: 2.1637
Pre-training Epoch [200/200], Loss: 2.0522
✅ Pretraining Completed
Entrenando a población sesgada
Entrenamiento finalizado para población sesgada
---------

### Modelo Recomendador #3: Mecanismo de Rankeo

Este modelo se basa en un enfoque de rankeo, donde los protocolos son evaluados y ordenados según su relevancia y adecuación al perfil del paciente. Utiliza un sistema de puntuación que considera múltiples variables clínicas y sociodemográficas para priorizar aquellos protocolos con mayor probabilidad de éxito. Al presentar un listado ordenado de recomendaciones, esta lógica permite al clínico seleccionar entre las opciones más adecuadas, manteniendo flexibilidad y control en la toma de decisiones.

In [ ]:
class RankingNN(nn.Module):
    def __init__(self, num_protocols, input_dim, priority_weights=None):
        super(RankingNN, self).__init__()
        self.input_dim = input_dim

        # Initialize attention weights based on priority
        if priority_weights is None:
            # Default priority: symptom > pathology > age > gender
            priority_weights = [0.35, 0.35, 0.15, 0.15]
            expanded_weights = (
                [priority_weights[0]] +  # Symptom
                [priority_weights[1]] +  # Pathology
                [priority_weights[2]] * 4 +  # Age groups
                [priority_weights[3]] * 3  # Gender groups
            )
        else:
            expanded_weights = priority_weights

        self.attention_weights = nn.Parameter(torch.tensor(expanded_weights, dtype=torch.float32), requires_grad=True)

        # Fully connected layers
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_protocols)

    def forward(self, x):
        # Apply attention weights to the input
        attention_scores = torch.softmax(self.attention_weights, dim=0)
        x = x * attention_scores

        # Forward pass through the network
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

class RankingRecommender:
    def __init__(self, model, label_encoder, df_protocols, pathology_keyword_mapping, symptom_keyword_mapping):
        self.model = model
        self.label_encoder = label_encoder
        self.df_protocols = df_protocols
        self.pathology_keyword_mapping = pathology_keyword_mapping
        self.symptom_keyword_mapping = symptom_keyword_mapping
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.0001)
        self.criterion = nn.CrossEntropyLoss(reduction='none')

    def recommend_protocol_training(self, patient_state, feedback_history_patient, feedback_history_clinician, rejected_protocols, current_protocol):
        """ENTRENAMIENTO: Recomienda un protocolo basado en el estado del paciente y actualiza el modelo."""

        self.model.eval()

        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if current_protocol is not None and (feedback_history_patient[-1] + feedback_history_clinician[-1])/2 > 4:
            return current_protocol
        else:
            rejected_protocols.add(current_protocol)

        with torch.no_grad():
            patient_state = patient_state.unsqueeze(0) if patient_state.dim() == 1 else patient_state
            outputs = self.model(patient_state)
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        while protocol_label in rejected_protocols:
            outputs[0, predicted_label_idx] = -float('inf')
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        return protocol_label


    def map_age_to_columns(self, age):
        age_groups = {'Kid_0_5': 0, 'Youth_6_17': 0, 'Adult_18_59': 0, 'Elderly_60_100': 0}
        if 0 <= age <= 5:
            age_groups['Kid_0_5'] = 1
        elif 6 <= age <= 17:
            age_groups['Youth_6_17'] = 1
        elif 18 <= age <= 59:
            age_groups['Adult_18_59'] = 1
        elif 60 <= age <= 100:
            age_groups['Elderly_60_100'] = 1
        return age_groups

    def map_gender_to_columns(self, gender):
        gender_columns = {'Males': 0.0, 'Females': 0.0, 'No_gender': 0.0}
        if gender.lower() == 'male':
            gender_columns['Males'] = 1.0
        elif gender.lower() == 'female':
            gender_columns['Females'] = 1.0
        else:
            gender_columns['No_gender'] = 1.0
        return gender_columns

    def generate_training_data_from_synthetic(self, synthetic_dataframes, num_sessions=10, training_poblacion_sesgada=False, training_poblacion_respuesta=False):
        """Genera datos de entrenamiento a partir de una población sintética."""
        training_data = []
        synthetic_dataframes = synthetic_dataframes.reset_index(drop=True)
        synthetic_dataframes['Response_profile'] = synthetic_dataframes['Response_profile'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


        for idx, user in synthetic_dataframes.iterrows():
            pathology_map = self.pathology_keyword_mapping[user['Pathology']]
            symptom_map = self.symptom_keyword_mapping[user['Symptom'].lower()]
            age_group_map = self.map_age_to_columns(user['Age'])
            gender_map = self.map_gender_to_columns(user['Gender'])

            inputs = [symptom_map, pathology_map] + list(age_group_map.values()) + list(gender_map.values())
            patient_inputs_tensor = torch.tensor(inputs, dtype=torch.float32)

            feedback_history_patient = []
            feedback_history_clinician = []
            rejected_protocols = set()

            user_response = user['Response_profile']

            for session in range(num_sessions):

                protocol_label = self.recommend_protocol_training(
                    patient_inputs_tensor,
                    feedback_history_patient,
                    feedback_history_clinician,
                    rejected_protocols,
                    None if session == 0 else protocol_label
                )

                if training_poblacion_sesgada:
                    feedback_score = self.check_protocol_feedback(protocol_label, synthetic_dataframes, idx)

                else:
                    print('entro al lugar equivocado')
                    feedback_score = int(user_response[session])

                feedback_history_patient.append(feedback_score)
                feedback_history_clinician.append(feedback_score)

                training_data.append((idx,patient_inputs_tensor, protocol_label, feedback_score))

                if session % 10 == 0 and session !=0 :
                    self.train_model_with_synthetic_data(training_data,synthetic_dataframes)

        return

    def train_model_with_synthetic_data(self, training_data, synthetic_dataframes, learning_rate=0.0001):
        self.model.train()
        batch_size = min(64, len(training_data))
        batch = random.sample(training_data, batch_size)

        epoch_loss = 0
        for idx, inputs, _,_ in batch:
            self.optimizer.zero_grad()
            inputs = inputs.unsqueeze(0) if inputs.dim() == 1 else inputs
            outputs = self.model(inputs)  # shape: [1, num_classes]

            predicted_label = torch.argmax(outputs, dim=1).item()
            feedback_score = self.check_protocol_feedback(predicted_label, synthetic_dataframes, idx)

            confidence = (feedback_score - 1) / 4.0  # 1 → 0.0, 5 → 1.0

            num_classes = self.model.fc4.out_features

            target_distribution = torch.full((1, num_classes), (1 - confidence) / (num_classes - 1))
            target_distribution[0, predicted_label] = confidence

            log_probs = torch.log_softmax(outputs, dim=1)
            loss = torch.nn.functional.kl_div(log_probs, target_distribution, reduction='batchmean')

            if feedback_score >= 4:
                penalty_factor = 0.2  # No scaling for good feedback
            else:
                penalty_factor = 1.1 + ((4 - feedback_score) / 3) ** 2  # e.g., 1.0 to ~2.78

            loss *= penalty_factor

            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()


    def incremental_training(self, synthetic_dataframes, num_epochs_per_stage=10, learning_rate=0.0001,training_poblacion_respuesta=False, training_poblacion_sesgada=False):

          if training_poblacion_respuesta:
              print('Entrenando a población respuesta')
              self.generate_training_data_from_synthetic(synthetic_dataframes, training_poblacion_respuesta=True)
              i = 'poblacion respuesta'

          elif training_poblacion_sesgada:
              print('Entrenando a población sesgada')
              self.generate_training_data_from_synthetic(synthetic_dataframes, training_poblacion_sesgada=True)
              i = 'población sesgada'

          print(f"Entrenamiento finalizado para {i}")
          print("----------------------------------------------------------")


    def check_protocol_feedback(self, recommended_protocol, synthetic_dataframes, patient_index):

        """
        Función de comprobación de protocolos.

        Arguments:
            - recommended_protocol (int): El corpusid del protocolo recomendado.
            - df_SintUsers (df): DataFrame con los datos de los pacientes de entrenamiento.
            - patient_index (int): El índice del paciente en el DataFrame.

        Returns:
            int: Feedback del paciente al protocolo (1-5).
        """
        synthetic_dataframes['Protocols'] = synthetic_dataframes['Protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        synthetic_dataframes['Exact_protocols'] = synthetic_dataframes['Exact_protocols'].apply(
            lambda x: x if isinstance(x, list) else ast.literal_eval(x)
        )

        # Extraemos los protocolos válidos y exactos del paciente
        valid_protocols = synthetic_dataframes.iloc[patient_index]['Protocols']
        extra_valid_protocols = synthetic_dataframes.iloc[patient_index]['Exact_protocols']

        # Asegurar que los valores son listas
        if not isinstance(valid_protocols, list):
            raise ValueError(f"Expected a list for 'Protocols', but got {type(valid_protocols)}")
        if not isinstance(extra_valid_protocols, list):
            raise ValueError(f"Expected a list for 'Exact_protocols', but got {type(extra_valid_protocols)}")

        # Si el protocolo recomendado está entre los válidos -- trata el síntoma
        if recommended_protocol in valid_protocols:
            # Si el protocolo trata también la patología del paciente
            if recommended_protocol in extra_valid_protocols:

                return 5.0  # Feedback positivo con patología y síntoma en común
            else:
                return 4.0  # Feedback positivo, pero sólo ajusta al síntoma

        # Si no está en la lista de protocolos válidos -- feedback negativo
        else:
            return random.choice([1.0, 2.0])

def pretrain_ranking_model(model, X, y, num_epochs=100, learning_rate=0.001):

    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y.numpy()), y=y.numpy())
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X)  # [batch_size, num_classes]
        loss = criterion(outputs, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"Pre-training Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

    print("✅ Pretraining complete!")



In [ ]:
input_columns = ['Symptom_map','Pathology_map', 'Kid_0_5', 'Youth_6_17', 'Adult_18_59',
                 'Elderly_60_100', 'Males', 'Females', 'No_gender']

protocol_columns = ['Current_map', 'Duration (min)', 'Times_per_day', 'Days_per_week',
                    'Cathode_standarized_map', 'Anode_standarized_map']

df_map[input_columns] = df_map[input_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
df_map[protocol_columns] = df_map[protocol_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode Protocol_Label to numeric labels
label_encoder = LabelEncoder()
df_map['Protocol_Label_encoded'] = label_encoder.fit_transform(df_map['Protocol_Label'])

X = torch.tensor(df_map[input_columns].values, dtype=torch.float32)
y = torch.tensor(df_map['Protocol_Label_encoded'].values, dtype=torch.long)

# Crear y pre-entrenar el modelo
state_dim = len(input_columns)
action_dim = len(label_encoder.classes_)
model_rank = RankingNN(action_dim,state_dim)
recommender_rank = RankingRecommender(model_rank, label_encoder, df_map, pathology_keyword_mapping, symptom_keyword_mapping)

save_path = "/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/"
os.makedirs(save_path, exist_ok=True)  # Ensure the directory exists

def save_final_model_weights(model, filename="tdcs_model_weights_ranking.pth"):
    """ Saves the final model weights after all training stages. """
    file_path = os.path.join(save_path, filename)
    torch.save(model.state_dict(), file_path)
    print(f"Final model weights saved at: {file_path}")

print("\n--- Iniciando Pre-Entrenamiento ---")
pretrain_ranking_model(model_rank, X, y)

train_val_splits_poblacion_respuesta = {}
train_val_splits_poblacion_sesgada = {}

train_df, val_df = train_test_split(poblacion_sesgada_completa, test_size=0.2, random_state=42)
train_val_splits_poblacion_sesgada = {"train": train_df, "val": val_df}

synthetic_training_dataframe_sesgada = train_val_splits_poblacion_sesgada["train"]

recommender_rank.incremental_training(synthetic_training_dataframe_sesgada, num_epochs_per_stage=7,training_poblacion_sesgada=True)

save_final_model_weights(model_rank) # Uncomment to rewrite the model weights


--- Iniciando Pre-Entrenamiento ---
Pre-training Epoch [10/100], Loss: 5.1137
Pre-training Epoch [20/100], Loss: 5.0795
Pre-training Epoch [30/100], Loss: 4.9902
Pre-training Epoch [40/100], Loss: 4.8154
Pre-training Epoch [50/100], Loss: 4.5977
Pre-training Epoch [60/100], Loss: 4.3614
Pre-training Epoch [70/100], Loss: 4.1172
Pre-training Epoch [80/100], Loss: 3.8903
Pre-training Epoch [90/100], Loss: 3.6890
Pre-training Epoch [100/100], Loss: 3.5137
✅ Pretraining complete!
Entrenando a población sesgada
Entrenamiento finalizado para población sesgada
----------------------------------------------------------
Final model weights saved at: /content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/tdcs_model_weights_ranking.pth


## **Sección de Despliegue**

En esta sección se realiza el despliegue del sistema de recomendación, utilizando los modelos previamente entrenados. A partir de los datos de entrada del paciente, el sistema carga los pesos correspondientes y ejecuta la lógica de recomendación para generar las sugerencias de protocolo personalizadas. Esta etapa no requiere reentrenamiento, por lo que es ideal para entornos de producción o pruebas clínicas donde se busca aplicar directamente las recomendaciones basadas en modelos ya optimizados.

### Modelo Recomendador #1: **Mecanismo de atención por Sintomatología**

In [ ]:
class AdaptiveTDCSNNWithAttention(nn.Module):
    def __init__(self, num_protocols, input_dim, priority_weights=None):
        super(AdaptiveTDCSNNWithAttention, self).__init__()
        self.input_dim = input_dim

        # Initialize attention weights based on priority
        if priority_weights is None:
            # Default priority: symptom > pathology > age > gender
            priority_weights = [0.6, 0.2, 0.1, 0.1]
            expanded_weights = (
                [priority_weights[0]] +  # Symptom
                [priority_weights[1]] +  # Pathology
                [priority_weights[2]] * 4 +  # Age groups
                [priority_weights[3]] * 3  # Gender groups
            )
        else:
            expanded_weights = priority_weights

        self.attention_weights = nn.Parameter(torch.tensor(expanded_weights, dtype=torch.float32), requires_grad=True)

        # Fully connected layers
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_protocols)

    def forward(self, x):
        # Apply attention weights to the input
        attention_scores = torch.softmax(self.attention_weights, dim=0)
        x = x * attention_scores

        # Forward pass through the network
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

class TDCSRecommenderWithAttention:
    def __init__(self, model_with_attention, label_encoder, df, pathology_keyword_mapping, symptom_keyword_mapping):
        self.model = model_with_attention
        self.label_encoder = label_encoder
        self.df = df
        self.pathology_keyword_mapping = pathology_keyword_mapping
        self.symptom_keyword_mapping = symptom_keyword_mapping
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)

    def calculate_weighted_feedback(self, feedback_history, decay_factor=0.8):
        min_feedback, max_feedback = 1, 5
        normalized_feedback = [(f - min_feedback) / (max_feedback - min_feedback) for f in feedback_history]
        feedback_weights = [decay_factor ** i for i in range(len(normalized_feedback))]
        weighted_feedback = sum(f * w for f, w in zip(normalized_feedback[::-1], feedback_weights))
        return weighted_feedback / sum(feedback_weights)

    def recommend_protocol_attention(self, inputs, feedback_history_patient, feedback_history_clinician, rejected_protocols, previous_protocol):
        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if previous_protocol is not None:
            weighted_patient_feedback = self.calculate_weighted_feedback(feedback_history_patient)
            weighted_clinician_feedback = self.calculate_weighted_feedback(feedback_history_clinician)

            weighted_total = (weighted_patient_feedback + weighted_clinician_feedback) / 2

            if weighted_total >= 0.5:
                return previous_protocol

            rejected_protocols.add(previous_protocol)

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(inputs)
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        while protocol_label in rejected_protocols:
            outputs[0, predicted_label_idx] = -float('inf')
            predicted_label_idx = torch.argmax(outputs, dim=1).item()
            protocol_label = self.label_encoder.inverse_transform([predicted_label_idx])[0]

        return protocol_label

### Modelo Recomendador #2: **Aprendizaje por Refuerzo**

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)  # Q-values for each protocol

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

class DQNAgent:
    def __init__(self, state_dim, action_dim, label_encoder, pathology_mapping, symptom_mapping,
                 learning_rate=0.001, gamma=0.7, memory_size=20000, batch_size=128):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.label_encoder = label_encoder
        self.pathology_mapping = pathology_mapping
        self.symptom_mapping = symptom_mapping

        # Initialize Q-network and target network
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.target_model.load_state_dict(self.model.state_dict())  # Sync target model
        self.last_feedback = 3.0

        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.MSELoss()

        # Experience replay buffer
        self.memory = deque(maxlen=memory_size)
        self.batch_size = batch_size

        # Hyperparameters
        self.gamma = gamma
        self.update_target_every = 3  # Update target network every X steps
        self.steps = 0

        self.loss_history = []  # Stores loss per batch

    def store_experience(self, state, action, reward, next_state):
        """Stores experiences in the replay buffer"""
        self.memory.append((state, action, reward, next_state))

    def select_action(self, state, previous_action,rejected_protocols,feedback_history_patient,feedback_history_clinician,generate_training=False):
        """Selects action using epsilon-greedy strategy"""

        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if previous_action is not None:
            if generate_training:
                if feedback_history_patient[-1] > 5:
                   action_idx = self.label_encoder.transform([previous_action])[0]
                   return int(action_idx), previous_action

                else:
                    rejected_protocols.add(previous_action)

            else: # Manual Checking
                if (feedback_history_patient[-1] + feedback_history_clinician[-1])/2 > 3:
                    action_idx = self.label_encoder.transform([previous_action])[0]
                    return int(action_idx), previous_action
                else:
                    rejected_protocols.add(previous_action)

        self.model.eval()
        with torch.no_grad():
              q_values = self.model(state)
              action_idx = torch.argmax(q_values).item()
              protocol_label = self.label_encoder.inverse_transform([action_idx])[0]

        while protocol_label in rejected_protocols:
              q_values[0, action_idx] = -float('inf')
              action_idx = torch.argmax(q_values).item()
              protocol_label = self.label_encoder.inverse_transform([action_idx])[0]

        return action_idx, protocol_label

    def train_RL(self,manual_comparison=False):
        """Trains the DQN using replay memory with constrained Q-value updates and feedback-aware loss weighting."""

        if len(self.memory) < self.batch_size:
            batch_size = len(self.memory)  # Use all available memory if smaller
        else:
            batch_size = self.batch_size

        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states = zip(*batch)

        states = torch.stack(states)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.stack(next_states)

        #Compute Q-values for selected actions
        q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            best_actions = self.target_model(next_states).argmax(1, keepdim=True)
            next_q_values = self.target_model(next_states).gather(1, best_actions).squeeze(1)

        # Reward Scaling
        structured_rewards = -2 + 4 * (rewards - 1) / 4  # Range [-2, 2]

        # Compute target Q-values with constraints
        target_q_values = structured_rewards + self.gamma * next_q_values

        # Feedback-Weighted Loss (low feedback → higher impact on training)
        loss_weights = torch.exp(-structured_rewards)  # More penalty for bad feedback

        if hasattr(self, 'last_feedback') and isinstance(self.last_feedback, (int, float)):
            if self.last_feedback <= 2:
                loss_weights *= 3  # More weight (higher penalty)

        weighted_loss = loss_weights * (q_values - target_q_values) ** 2
        loss = weighted_loss.mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()

        self.loss_history.append(loss.item())

        # Update target network periodically
        if manual_comparison:
            self.target_model.load_state_dict(self.model.state_dict())
        else:
            if self.steps % self.update_target_every == 0:
                self.target_model.load_state_dict(self.model.state_dict())

        self.steps += 1


### Modelo Recomendador #3: Mecanismo de Rankeo

In [ ]:
class RankingNN(nn.Module):
    def __init__(self, num_protocols, input_dim, priority_weights=None):
        super(RankingNN, self).__init__()
        self.input_dim = input_dim

        # Initialize attention weights based on priority
        if priority_weights is None:
            # Default priority: symptom > pathology > age > gender
            priority_weights = [0.35, 0.35, 0.15, 0.15]
            expanded_weights = (
                [priority_weights[0]] +  # Symptom
                [priority_weights[1]] +  # Pathology
                [priority_weights[2]] * 4 +  # Age groups
                [priority_weights[3]] * 3  # Gender groups
            )
        else:
            expanded_weights = priority_weights

        self.attention_weights = nn.Parameter(torch.tensor(expanded_weights, dtype=torch.float32), requires_grad=True)

        # Fully connected layers
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_protocols)

    def forward(self, x):
        # Apply attention weights to the input
        attention_scores = torch.softmax(self.attention_weights, dim=0)
        x = x * attention_scores

        # Forward pass through the network
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

class RankingRecommender:
    def __init__(self, model_with_rank, label_encoder, df_protocols, pathology_keyword_mapping, symptom_keyword_mapping):
        self.model = model_with_rank
        self.label_encoder = label_encoder
        self.df = df_protocols
        self.pathology_keyword_mapping = pathology_keyword_mapping
        self.symptom_keyword_mapping = symptom_keyword_mapping
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.criterion = nn.CrossEntropyLoss()

    def recommend_protocol_ranking(self, patient_state, feedback_history_patient, feedback_history_clinician, rejected_protocols, current_protocol):
        """Recomienda un protocolo basado en el estado del paciente y feedback."""
        self.model.eval()

        if len(rejected_protocols) >= 10:
            print("Clearing rejected protocols list to allow reuse of previously rejected protocols.")
            rejected_protocols.clear()

        if current_protocol is not None and (feedback_history_patient[-1] + feedback_history_clinician[-1])/2 > 4:
            return current_protocol
        else:
            rejected_protocols.add(current_protocol)

        with torch.no_grad():
            patient_state = patient_state.unsqueeze(0) if patient_state.dim() == 1 else patient_state
            protocol_logits = self.model(patient_state)
            protocol_probs = torch.softmax(protocol_logits, dim=1)
            sorted_indices = torch.argsort(protocol_probs, descending=True).tolist()[0]


        if isinstance(sorted_indices[0], list):
            sorted_indices = [protocol for sublist in sorted_indices for protocol in sublist]

        # Mostrar el ranking de protocolos al clínico
        for item in rejected_protocols:
            if item in sorted_indices:
                sorted_indices.remove(item)

        top_protocols = sorted_indices[:3]
        top_protocol_labels = self.label_encoder.inverse_transform(top_protocols)

        print("\nTop 3 Protocols:")
        for idx, protocol_label in enumerate(top_protocol_labels):
            print('----------------------------------------')
            print(f"{idx + 1}. Protocol {protocol_label}")
            print('----------------------------------------')
            protocol_row = df_no_map.loc[df_no_map['Protocol_Label'] == protocol_label].iloc[0]

            cathode_final = (protocol_row['Cathode']
                             if protocol_row['Cathode_standarized'] in ['others', 'Anatomical area']
                             else protocol_row['Cathode_standarized'])

            anode_final = (protocol_row['Anode']
                           if protocol_row['Anode_standarized'] in ['others', 'Anatomical area']
                           else protocol_row['Anode_standarized'])

            protocol_parameters = {
                  '- Current (mA)': protocol_row['Current (mA)'],
                  '- Duration (min)': protocol_row['Duration (min)'],
                  '- Times per day': protocol_row['Times_per_day'],
                  '- Days per week': protocol_row['Days_per_week'],
                  '- Cathode placement': cathode_final,
                  '- Anode placement': anode_final,
                  '- Modality' : protocol_row['Modality'],
                  '- Paper reference' : protocol_row['Title'],
                  '- URL': protocol_row['url']
              }

            print("Protocol Parameters:")
            for key, value in protocol_parameters.items():
                print(f"{key}: {value}")

        print('----------------------------------------')
        choice = (int(input("Select a protocol (1-3): ")) - 1)
        print('----------------------------------------')
        selected_protocol = top_protocol_labels[choice]
        next_protocol = selected_protocol

        return next_protocol


### **Sistema de Transferencia de Estados**

Este proceso actúa como un núcleo integrador entre los tres modelos de recomendación. Su función principal es permitir el intercambio de información relevante entre las lógicas, incluyendo el historial de retroalimentación del paciente, los protocolos previamente rechazados y las características clínicas y sociodemográficas del paciente. Gracias a esta transferencia de estados, los modelos pueden mantener una continuidad en la toma de decisiones, adaptándose de forma coherente al progreso de cada caso y mejorando la personalización de las recomendaciones a lo largo del tiempo.

In [ ]:
## Sistema de Transferencia de Estados

def map_age_to_columns(age):
        age_groups = {'Kid_0_5': 0, 'Youth_6_17': 0, 'Adult_18_59': 0, 'Elderly_60_100': 0}
        if 0 <= age <= 5:
            age_groups['Kid_0_5'] = 1
        elif 6 <= age <= 17:
            age_groups['Youth_6_17'] = 1
        elif 18 <= age <= 59:
            age_groups['Adult_18_59'] = 1
        elif 60 <= age <= 100:
            age_groups['Elderly_60_100'] = 1
        return age_groups

def map_gender_to_columns(gender):
    gender_columns = {'Males': 0.0, 'Females': 0.0, 'No_gender': 0.0}
    if gender.lower() == 'male':
        gender_columns['Males'] = 1.0
    elif gender.lower() == 'female':
        gender_columns['Females'] = 1.0
    else:
        gender_columns['No_gender'] = 1.0
    return gender_columns

def get_patient_data(pathology_keyword_mapping,symptom_keyword_mapping):
    print("Please select the patient's pathology:")
    for i, (key, value) in enumerate(pathology_keyword_mapping.items()):
        print(f"{i}. {key}")
    pathology_choice = int(input("Enter the number corresponding to the pathology: "))
    pathology_map = list(pathology_keyword_mapping.values())[pathology_choice]

    print("\nPlease select the patient's symptom:")
    for i, (key, value) in enumerate(symptom_keyword_mapping.items()):
        print(f"{i}. {key}")
    symptom_choice = int(input("Enter the number corresponding to the symptom: "))
    symptom_map = list(symptom_keyword_mapping.values())[symptom_choice]

    age = int(input("Enter the patient's exact age: "))
    gender = input("Enter the patient's gender (Male/Female/Other): ")

    age_groups = map_age_to_columns(age)
    gender_columns = map_gender_to_columns(gender)

    inputs = [symptom_map, pathology_map] + list(age_groups.values()) + list(gender_columns.values())
    inputs_tensor = torch.tensor([inputs], dtype=torch.float32)

    return inputs_tensor, pathology_map, symptom_map, age, gender

def get_session_feedback():
    while True:
        subjective_test = float(input("Enter the Subjective Test Score (1-5): "))
        clinician_feedback = float(input("Enter the Clinician Feedback (1-5): "))
        if 0 <= subjective_test <= 5 and 0 <= clinician_feedback <= 5:
            return subjective_test, clinician_feedback
        else:
            print("Please enter values within the range 0 to 5.")

def state_transfer_system(last_model=None):
    valid_models = {1: 'attention', 2: 'reinforcement_learning', 3: 'ranking'}

    if last_model is None:
        while True:
            try:
                model_type = int(input(
                    "Choose the logic to use:\n"
                    "1 - Attention Mechanism\n"
                    "2 - Reinforcement Learning\n"
                    "3 - Ranking Mechanism\n"
                    "Enter your choice (1/2/3): "
                ))
                if model_type in valid_models:
                    break
                else:
                    print("Invalid choice. Please select 1, 2, or 3.")
            except ValueError:
                print("Please enter a number (1, 2, or 3).")

    else:
        while True:
            response = input(f"Continue with {last_model}? (Yes/No): ").strip().lower()
            if response == 'no':
                while True:
                    try:
                        model_type = int(input(
                            "Choose the logic to use:\n"
                            "1 - Attention Mechanism\n"
                            "2 - Reinforcement Learning\n"
                            "3 - Ranking Mechanism\n"
                            "Enter your choice (1/2/3): "
                        ))
                        if model_type in valid_models:
                            break
                        else:
                            print("Invalid choice. Please select 1, 2, or 3.")
                    except ValueError:
                        print("Please enter a number (1, 2, or 3).")
                break

            elif response == 'yes':
                model_type = [key for key, value in valid_models.items() if value == last_model][0]
                break

            else:
                print("Invalid response. Please enter 'yes' or 'no'.")

    return model_type

**CARGAR MODELOS Y SUS PESOS**

In [ ]:
input_columns = ['Symptom_map','Pathology_map', 'Kid_0_5', 'Youth_6_17', 'Adult_18_59',
                 'Elderly_60_100', 'Males', 'Females', 'No_gender']

protocol_columns = ['Current_map', 'Duration (min)', 'Times_per_day', 'Days_per_week',
                    'Cathode_standarized_map', 'Anode_standarized_map']

df_map[input_columns] = df_map[input_columns].apply(pd.to_numeric, errors='coerce').fillna(0)
df_map[protocol_columns] = df_map[protocol_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode Protocol_Label to numeric labels
label_encoder = LabelEncoder()
df_map['Protocol_Label_encoded'] = label_encoder.fit_transform(df_map['Protocol_Label'])

X = torch.tensor(df_map[input_columns].values, dtype=torch.float32)
y = torch.tensor(df_map['Protocol_Label_encoded'].values, dtype=torch.long)

num_protocols = len(label_encoder.classes_)
input_dim = len(input_columns)

### Loading Ranking Mechanism Model
model_with_rank = RankingNN(num_protocols,input_dim)

# Path to the saved weights - INTRODUCIR AQUI PATH DE LOS PESOS PARA RANKING
save_path_ranking = Path('/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/tdcs_model_weights_ranking.pth')
# Load the weights from the file
loaded_weights_ranking = torch.load(save_path_ranking)
model_with_rank.load_state_dict(loaded_weights_ranking)  # cambiar de acuerdo al nombre del modelo guardado
model_with_rank.eval()
print('---------------------------------------------------------------')
print("Weights loaded into model_with_ranking.")

# Initialize the Recommender_RL class with the loaded model
recommender_ranking = RankingRecommender(
    model_with_rank = model_with_rank,
    label_encoder=label_encoder,
    df_protocols=df_no_map,
    pathology_keyword_mapping=pathology_keyword_mapping,
    symptom_keyword_mapping=symptom_keyword_mapping
)

print("Recommender_Ranking initialized successfully.")
### Loading Reinforced Learning Model
agent = DQNAgent(state_dim=input_dim, action_dim=num_protocols, label_encoder=label_encoder, pathology_mapping=pathology_keyword_mapping, symptom_mapping=symptom_keyword_mapping)

# Path to the saved weights
save_path_RL = Path('/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/tdcs_model_weights_RL.pth')
# Load the weights from the file
loaded_weights_RL = torch.load(save_path_RL)
agent.model.load_state_dict(loaded_weights_RL)
agent.target_model.load_state_dict(agent.model.state_dict())
agent.model.eval()
agent.target_model.eval()

print('---------------------------------------------------------------')
print("Weights loaded into model_with_RL.")
print("Recommender_RL initialized successfully.")

## Loading Attention Mechanism Model
model_with_attention = AdaptiveTDCSNNWithAttention(num_protocols, input_dim)

# Path to the saved weights
save_path_attention = Path('/content/drive/MyDrive/Data Lake IONClinics/Pesos_modelos/tdcs_model_weights_attention.pth')
# Load the weights from the file
loaded_weights_attention = torch.load(save_path_attention)
model_with_attention.load_state_dict(loaded_weights_attention)
model_with_attention.eval()
print('---------------------------------------------------------------')
print("Weights loaded into model_with_attention.")

# Initialize the Recommender_RL class with the loaded model
recommender_attention = TDCSRecommenderWithAttention(
    model_with_attention=model_with_attention,
    label_encoder=label_encoder,
    df=df_no_map,
    pathology_keyword_mapping=pathology_keyword_mapping,
    symptom_keyword_mapping=symptom_keyword_mapping
)

print("Recommender_attention initialized successfully.")

---------------------------------------------------------------
Weights loaded into model_with_ranking.
Recommender_Ranking initialized successfully.
---------------------------------------------------------------
Weights loaded into model_with_RL.
Recommender_RL initialized successfully.
---------------------------------------------------------------
Weights loaded into model_with_attention.
Recommender_attention initialized successfully.


In [ ]:
def deployment(pathology_keyword_mapping, symptom_keyword_mapping):
    # Get patient data
    inputs_tensor, pathology_map, symptom_map, age, gender = get_patient_data(pathology_keyword_mapping, symptom_keyword_mapping)

    # Reverse Mapping for Readable Pathology & Symptom Names
    pathology_name = reverse_pathology_keyword_mapping.get(pathology_map, "Unknown")
    symptom_name = reverse_symptom_keyword_mapping.get(symptom_map, "Unknown")
    patient_ID = str(uuid.uuid4())

    print("\n" + "="*50)
    print("                PATIENT INFORMATION")
    print("="*50)
    print(f"Patient ID: {str(patient_ID)}")
    print(f"Pathology: {pathology_name}")
    print(f"Symptom: {symptom_name}")
    print(f"Age: {age} years")
    print(f"Gender: {gender}")
    print("="*50 + "\n")

    # Initialize patient record (moved outside the loop to persist across additional sessions)
    patient_record = {
        'patient_ID': patient_ID,
        'age': age,
        'gender': gender,
        'pathology': pathology_name,
        'symptom': symptom_name,
        'sessions': []  # List to store data for each session
    }

    feedback_history_patient = []
    feedback_history_clinician = []
    rejected_protocols = set()
    protocol_label = None
    last_model = None

    # Start session count at 1
    session_counter = 1
    more_sessions = True

    while more_sessions:
        num_sessions = int(input("Enter the number of sessions: "))

        for _ in range(num_sessions):
            print(f"\nSession {session_counter}")
            session_data = {'session_number': session_counter}

            if session_counter > 1:
                subjective_test, clinician_feedback = get_session_feedback()
                feedback_history_patient.append(subjective_test)
                feedback_history_clinician.append(clinician_feedback)
                session_data['subjective_test'] = subjective_test
                session_data['clinician_feedback'] = clinician_feedback
            else:
                session_data['subjective_test'] = None  # No feedback for the first session
                session_data['clinician_feedback'] = None

            # Get the selected model type
            if last_model == 'reinforcement_learning':
                agent.last_feedback = feedback_history_clinician[-1]
                state_tensor = inputs_tensor.squeeze(0)
                next_state = state_tensor
                agent.store_experience(state_tensor, protocol_label, feedback_history_clinician[-1], next_state)

                agent.train_RL(manual_comparison=True)


            model_type = state_transfer_system(last_model)

            if model_type == 1:
                print("--------------------------")
                print("Using Attention Mechanism")
                protocol_label = recommender_attention.recommend_protocol_attention(
                    inputs_tensor,
                    feedback_history_patient,
                    feedback_history_clinician,
                    rejected_protocols,
                    protocol_label
                )
                last_model = 'attention'
                session_data['model_type'] = 'Attention Mechanism'

            elif model_type == 2:
                print("--------------------------")
                print("Using Reinforcement Learning")
                _,protocol_label = agent.select_action(
                    state=inputs_tensor,
                    previous_action=protocol_label,
                    rejected_protocols=rejected_protocols,
                    feedback_history_patient=feedback_history_patient,
                    feedback_history_clinician=feedback_history_clinician,)

                last_model = 'reinforcement_learning'
                session_data['model_type'] = 'Reinforcement Learning'

            elif model_type == 3:
                print("--------------------------")
                print("Using Ranking Mechanism")
                protocol_label = recommender_ranking.recommend_protocol_ranking(
                    inputs_tensor,
                    feedback_history_patient,
                    feedback_history_clinician,
                    rejected_protocols,
                    protocol_label
                )
                last_model = 'ranking'
                session_data['model_type'] = 'Ranking Mechanism'

            # Add protocol label to session data
            session_data['protocol_label'] = protocol_label
            print(f"Suggested Protocol: {protocol_label}")

            matching_protocols = df_no_map[df_no_map['Protocol_Label'] == protocol_label]

            # Extract consistent protocol parameters from the first matching row
            first_row = matching_protocols.iloc[0]
            cathode_final = (first_row['Cathode']
                            if first_row['Cathode_standarized'] in ['others', 'Anatomical area']
                            else first_row['Cathode_standarized'])

            anode_final = (first_row['Anode']
                          if first_row['Anode_standarized'] in ['others', 'Anatomical area']
                          else first_row['Anode_standarized'])

            protocol_parameters = {
                "- Current (mA)": first_row['Current (mA)'],
                "- Duration (min)": first_row['Duration (min)'],
                "- Times per day": first_row['Times_per_day'],
                "- Days per week": first_row['Days_per_week'],
                "- Cathode placement": cathode_final,
                "- Anode placement": anode_final,
                "- Modality": first_row['Modality'],
            }

            print("Protocol Parameters:")
            for key, value in protocol_parameters.items():
                print(f"{key}: {value}")

            print("\nAssociated Publications:")
            for i, (_, row) in enumerate(matching_protocols.iterrows(), start=1):
                title = row['Title']
                url = row['url']
                print(f"- Paper reference ({i}): {title}")
                print(f"- URL ({i}): {url}\n")


            # Add session data to the patient record
            patient_record['sessions'].append(session_data)
            print("----------------------------------------------------------")

            # Increment session counter
            session_counter += 1

        # Ask if user wants to continue
        additional_sessions = input("Do you want to continue with more sessions? (yes/no): ").strip().lower()
        more_sessions = additional_sessions == 'yes'

    print("\nFinal Patient Record:")
    return patient_record

## Sección de Test

En esta última etapa se implementa la función de integración del sistema de recomendación y se habilita la interacción manual por parte del clínico. Aquí, el profesional introduce las características del paciente (como patología, síntomas, edad, género) y registra la retroalimentación tras cada sesión terapéutica. Esta sección permite evaluar el comportamiento del sistema en un entorno simulado, facilitando el análisis de la respuesta de los modelos ante diferentes perfiles y progresos clínicos.

In [ ]:
## Depression - Depression
patient_record = deployment(pathology_keyword_mapping,symptom_keyword_mapping)

Please select the patient's pathology:
0. Not specified
1. others
2. stroke
3. pain
4. depression
5. schizophrenia
6. spinal cord injury
7. fibromyalgia
8. cognitive decline
9. knee osteoarthritis
10. mental-health disorder
11. cancer
Enter the number corresponding to the pathology: 4

Please select the patient's symptom:
0. Not specified
1. others
2. motor
3. depression
4. chronic pain
5. aphasia
6. complex symptoms associated with fibromyalgia
7. cognitive function
8. neuropathic pain
9. pain associated with fibromyalgia
10. complex symptoms associated with schizophrenia
11. memory
Enter the number corresponding to the symptom: 3
Enter the patient's exact age: 50
Enter the patient's gender (Male/Female/Other): female

                PATIENT INFORMATION
Patient ID: 92086544-3028-4fdf-ac81-b85bac3ca775
Pathology: depression
Symptom: depression
Age: 50 years
Gender: female

Enter the number of sessions: 20

Session 1
Choose the logic to use:
1 - Attention Mechanism
2 - Reinforcement Le

In [ ]:
## Stroke - Aphasia
patient_record2 = deployment(pathology_keyword_mapping,symptom_keyword_mapping)

Please select the patient's pathology:
0. Not specified
1. others
2. stroke
3. pain
4. depression
5. schizophrenia
6. spinal cord injury
7. fibromyalgia
8. cognitive decline
9. knee osteoarthritis
10. mental-health disorder
11. cancer
Enter the number corresponding to the pathology: 2

Please select the patient's symptom:
0. Not specified
1. others
2. motor
3. depression
4. chronic pain
5. aphasia
6. complex symptoms associated with fibromyalgia
7. cognitive function
8. neuropathic pain
9. pain associated with fibromyalgia
10. complex symptoms associated with schizophrenia
11. memory
Enter the number corresponding to the symptom: 5
Enter the patient's exact age: 60
Enter the patient's gender (Male/Female/Other): male

                PATIENT INFORMATION
Patient ID: 6ad6f0f5-fe57-43cf-89f6-c54b1cef39fe
Pathology: stroke
Symptom: aphasia
Age: 60 years
Gender: male

Enter the number of sessions: 20

Session 1
Choose the logic to use:
1 - Attention Mechanism
2 - Reinforcement Learning
3 - 

ValueError: could not convert string to float: ''